In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/SriramSatvik-dev/defectlens.git
%cd defectlens

Cloning into 'defectlens'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 57 (delta 19), reused 48 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 32.53 KiB | 16.27 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/defectlens


In [3]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = '/content/drive/MyDrive/defectlens/.kaggle'

In [4]:
!mkdir -p /content/mvtec_raw
print("Done")
!cp -r /content/drive/MyDrive/defectlens/data/screw /content/mvtec_raw/
print("Done")
!cp -r /content/drive/MyDrive/defectlens/data/pill /content/mvtec_raw/
print("Done")

Done
Done
Done


In [ ]:
!ls /content/mvtec_raw/screw
!ls /content/mvtec_raw/screw/train
!ls /content/mvtec_raw/screw/test
!ls /content/mvtec_raw/screw/ground_truth

!ls /content/mvtec_raw/pill
!ls /content/mvtec_raw/pill/train
!ls /content/mvtec_raw/pill/test
!ls /content/mvtec_raw/pill/ground_truth

ground_truth  license.txt  readme.txt  test  train
good
good  manipulated_front  scratch_head  scratch_neck  thread_side  thread_top
manipulated_front  scratch_head  scratch_neck  thread_side  thread_top
ground_truth  license.txt  readme.txt  test  train
good
color  combined  contamination	crack  faulty_imprint  good  pill_type	scratch
color  combined  contamination	crack  faulty_imprint  pill_type  scratch


In [ ]:
# import os

for cat in ['screw', 'pill']:
    train_good = len(os.listdir(f'/content/mvtec_raw/{cat}/train/good'))
    test_good = len(os.listdir(f'/content/mvtec_raw/{cat}/test/good'))
    defect_dirs = [d for d in os.listdir(f'/content/mvtec_raw/{cat}/test') if d != 'good']
    defect_counts = {d: len(os.listdir(f'/content/mvtec_raw/{cat}/test/{d}')) for d in defect_dirs}
    print(cat, '| train/good:', train_good, '| test/good:', test_good, '| test defects:', defect_counts)

screw | train/good: 320 | test/good: 41 | test defects: {'scratch_head': 24, 'manipulated_front': 24, 'thread_side': 23, 'thread_top': 23, 'scratch_neck': 25}
pill | train/good: 267 | test/good: 26 | test defects: {'faulty_imprint': 19, 'color': 25, 'scratch': 24, 'pill_type': 9, 'contamination': 21, 'combined': 17, 'crack': 26}


In [ ]:
!python -m src.data.dataset /content/mvtec_raw/ screw /content/drive/MyDrive/defectlens/splits

[screw] train: 320 | val: 24 | test: 136
x_encoder shape: torch.Size([3, 256, 256]) mean: 1.08125638961792
x_target shape: torch.Size([3, 256, 256]) min/max: 0.15294118225574493 0.8039215803146362
val sample label: 0 defect_type: good


In [ ]:
!python -m src.data.dataset /content/mvtec_raw/ pill /content/drive/MyDrive/defectlens/splits

[pill] train: 267 | val: 25 | test: 142
x_encoder shape: torch.Size([3, 256, 256]) mean: -0.6344830393791199
x_target shape: torch.Size([3, 256, 256]) min/max: 0.03529411926865578 0.8352941274642944
val sample label: 1 defect_type: color


In [ ]:
!python -m src.models.autoencoder

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 238MB/s]
Input shape: torch.Size([2, 3, 256, 256])
Output shape: torch.Size([2, 3, 256, 256])
Encoder params requiring grad: 0 (expect 0, frozen by default)
Decoder params requiring grad: 18 (expect >0, trained from scratch)
Output min/max: 0.13610298931598663 0.9275516271591187 (expect within [0,1])
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 217MB/s]
EfficientNet-B0 ablation path output shape: torch.Size([2, 3, 256, 256]) (matches input, as expected)


In [5]:
!pip install -q pytorch-msssim

In [ ]:
!python -m src.losses.mse_loss

MSE loss: 0.16658630967140198
MSE backward OK, recon.grad is not None: True
Identical-input sanity check -- MSE: 0.000000 (expect ~0)


In [ ]:
!python -m src.losses.ssim_composite_loss

Composite loss: 1.3266745805740356 | components: {'l1': 0.33315661549568176, 'ssim_term': 0.9935179352760315}
Composite backward OK, recon.grad is not None: True
Identical-input sanity check -- Composite: 0.000000 (expect ~0)


In [ ]:
!python -m src.eval.metrics

Max scores: [0.09970666 0.09998382 0.09991451 0.09991702 0.09971379 0.9
 0.9        0.9        0.9        0.9       ]
Mean scores: [0.04774771 0.05168226 0.05019775 0.05028925 0.04769111 0.05357878
 0.05365438 0.05455406 0.05300476 0.05099364]
AUC-ROC (max aggregation): 1.0000 (expect ~1.0 -- spike is very localized)
AUC-ROC (mean aggregation): 0.9600 (expect lower than max -- spike diluted by averaging)
Sanity check passed: max aggregation correctly more sensitive to localized defects here.


In [6]:
!pip install -q wandb

In [ ]:
!python -m src.training.freeze_utils
!python -m src.training.scheduler

After freeze=True init: 0 trainable params (expect 0)
After partial unfreeze -- stem: 0 (expect 0), layer1: 0 (expect 0), layer2: 0 (expect 0), layer3: 2099712 (expect >0)
Partial unfreeze verified: only layer3 is trainable, as configured.
Initial LR: 0.001
Epoch 1: val_auc=0.75, lr=0.001
Epoch 2: val_auc=0.75, lr=0.001
Epoch 3: val_auc=0.75, lr=0.001
Epoch 4: val_auc=0.75, lr=0.0005
Epoch 5: val_auc=0.75, lr=0.0005
Epoch 6: val_auc=0.75, lr=0.0005
Verified: LR dropped from 0.001 to 0.0005 after plateau.


In [ ]:
!python -m src.training.train --category screw --loss mse --epochs 2 --run_name smoke_test

Using device: cuda
[screw] train: 320 | val: 24 | test: 136
Encoder trainable params: 0 (freeze_encoder=True)
Epoch 1/2 | train_loss: 0.0421 | val_mse: 0.0226 | val_auc: 0.1389 | lr: 1.00e-03
  -> New best val_auc: 0.1389, checkpoint saved.
Epoch 2/2 | train_loss: 0.0049 | val_mse: 0.0040 | val_auc: 0.6944 | lr: 1.00e-03
  -> New best val_auc: 0.6944, checkpoint saved.
Training complete. Best val_auc: 0.6944. Checkpoint: /content/drive/MyDrive/defectlens/checkpoints/smoke_test_best.pt


In [7]:
from google.colab import userdata
import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!python -m src.training.train --category screw --loss mse --epochs 50 --run_name baseline_mse_screw --use_wandb

Using device: cuda
[screw] train: 320 | val: 24 | test: 136
Encoder trainable params: 0 (freeze_encoder=True)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run s9e9diur (0.0s)
wandb: ⣻ setting up run s9e9diur (0.0s)
wandb: ⣽ setting up run s9e9diur (0.0s)
wandb: ⣾ setting up run s9e9diur (0.0s)
wandb: ⣷ setting up run s9e9diur (0.5s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260805_162814-s9e9diur
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run baseline_mse_screw
wandb: ⭐️ View project at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens
wandb: 🚀 View run at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens/runs/

In [ ]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/baseline_mse_screw_best.pt

Loaded checkpoint from epoch 24 (val_auc at save time: 0.6667)
Config: loss=mse, freeze_encoder=True, encoder_backbone=resnet18
Test set size: 136

--- Aggregation: max ---
Overall test AUC-ROC: 0.8436
Per-defect-type AUC-ROC:
  thread_side: 0.7829
  scratch_head: 0.8314
  scratch_neck: 0.8408
  manipulated_front: 0.8529
  thread_top: 0.9100

--- Aggregation: mean ---
Overall test AUC-ROC: 0.0141
Per-defect-type AUC-ROC:
  thread_side: 0.0057
  scratch_head: 0.0129
  scratch_neck: 0.0150
  manipulated_front: 0.0186
  thread_top: 0.0186

Results saved to /content/drive/MyDrive/defectlens/results/baseline_mse_screw_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/screw/test/thread_top/014.png | defect_type: thread_top | score: 0.5076
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/screw/test/manipulated_front/017.png | defect_type: manipulated_front | score: 0.2475
Figure(1500x500)

Visualizing: example_good_i

In [ ]:
!python -m src.training.train --category pill --loss mse --epochs 50 --run_name baseline_mse_pill --use_wandb

Using device: cuda
[pill] train: 267 | val: 25 | test: 142
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 202MB/s]
Encoder trainable params: 0 (freeze_encoder=True)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260807_113737-2hmxxm5m
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run baseline_mse_pill
wandb: ⭐️ View project at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens
wa

In [ ]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/baseline_mse_pill_best.pt

Loaded checkpoint from epoch 10 (val_auc at save time: 0.8095)
Config: loss=mse, freeze_encoder=True, encoder_backbone=resnet18
Test set size: 142

--- Aggregation: max ---
Overall test AUC-ROC: 0.6947
Per-defect-type AUC-ROC:
  contamination: 0.5227
  scratch: 0.5364
  faulty_imprint: 0.6278
  crack: 0.7190
  color: 0.7684
  combined: 0.8734
  pill_type: 1.0000

--- Aggregation: mean ---
Overall test AUC-ROC: 0.8826
Per-defect-type AUC-ROC:
  crack: 0.7893
  scratch: 0.8477
  faulty_imprint: 0.8636
  color: 0.8658
  contamination: 0.9242
  combined: 0.9968
  pill_type: 1.0000

Results saved to /content/drive/MyDrive/defectlens/results/baseline_mse_pill_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/pill/test/color/006.png | defect_type: color | score: 0.7105
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/pill/test/color/002.png | defect_type: color | score: 0.2230
Figure(1500x500)

Visualizing: example_g

In [ ]:
!python -m src.training.train --category screw --loss composite --epochs 50 --lr 1e-4 --run_name ablation1_composite_screw_lowlr --use_wandb

Using device: cuda
[screw] train: 320 | val: 24 | test: 136
Encoder trainable params: 0 (freeze_encoder=True)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run 2eccsgwd (0.0s)
wandb: ⣻ setting up run 2eccsgwd (0.0s)
wandb: ⣽ setting up run 2eccsgwd (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260807_132436-2eccsgwd
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ablation1_composite_screw_lowlr
wandb: ⭐️ View project at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens
wandb: 🚀 View run at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens/runs/2eccsgwd
Epoch 1/50 | train_loss: 0.8029 | val_mse: 0.0616 | val_au

In [ ]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/ablation1_composite_screw_lowlr_best.pt

Loaded checkpoint from epoch 8 (val_auc at save time: 0.8704)
Config: loss=composite, freeze_encoder=True, encoder_backbone=resnet18
Test set size: 136

--- Aggregation: max ---
Overall test AUC-ROC: 0.8885
Per-defect-type AUC-ROC:
  manipulated_front: 0.7071
  scratch_neck: 0.9170
  thread_side: 0.9329
  thread_top: 0.9386
  scratch_head: 0.9457

--- Aggregation: mean ---
Overall test AUC-ROC: 1.0000
Per-defect-type AUC-ROC:
  manipulated_front: 1.0000
  scratch_head: 1.0000
  scratch_neck: 1.0000
  thread_side: 1.0000
  thread_top: 1.0000

Results saved to /content/drive/MyDrive/defectlens/results/ablation1_composite_screw_lowlr_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/screw/test/thread_side/002.png | defect_type: thread_side | score: 0.5174
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/screw/test/manipulated_front/003.png | defect_type: manipulated_front | score: 0.2612
Figure(1500x500)

Visuali

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.data.dataset import get_datasets
from src.eval.evaluate import rebuild_model_from_checkpoint, run_full_test_eval
from src.eval.metrics import aggregate_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    "/content/drive/MyDrive/defectlens/checkpoints/ablation1_composite_screw_lowlr_best.pt",
    map_location=device, weights_only=False
)
model, saved_args = rebuild_model_from_checkpoint(ckpt, device)

_, _, test_ds = get_datasets(saved_args["data_root"], saved_args["category"], saved_args["splits_dir"])
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

error_maps, x_targets, recons, labels, defect_types, paths = run_full_test_eval(model, test_loader, device)

mean_scores = aggregate_score(error_maps, method="mean").numpy()

for i in range(len(mean_scores)):
    print(f"label={labels[i]} | defect_type={defect_types[i]:18s} | mean_score={mean_scores[i]:.6f} | path={paths[i]}")

print("\n--- Summary ---")
print("Good mean:  ", mean_scores[labels==0].mean(), "| std:", mean_scores[labels==0].std())
print("Defect mean:", mean_scores[labels==1].mean(), "| std:", mean_scores[labels==1].std())
print("Min defect score:", mean_scores[labels==1].min(), "| Max good score:", mean_scores[labels==0].max())
print("Gap (min defect - max good):", mean_scores[labels==1].min() - mean_scores[labels==0].max())

label=0 | defect_type=good               | mean_score=0.099165 | path=/content/mvtec_raw/screw/test/good/030.png
label=0 | defect_type=good               | mean_score=0.101512 | path=/content/mvtec_raw/screw/test/good/011.png
label=0 | defect_type=good               | mean_score=0.111097 | path=/content/mvtec_raw/screw/test/good/004.png
label=0 | defect_type=good               | mean_score=0.107721 | path=/content/mvtec_raw/screw/test/good/032.png
label=0 | defect_type=good               | mean_score=0.108544 | path=/content/mvtec_raw/screw/test/good/021.png
label=0 | defect_type=good               | mean_score=0.103093 | path=/content/mvtec_raw/screw/test/good/028.png
label=0 | defect_type=good               | mean_score=0.112568 | path=/content/mvtec_raw/screw/test/good/023.png
label=0 | defect_type=good               | mean_score=0.102234 | path=/content/mvtec_raw/screw/test/good/018.png
label=0 | defect_type=good               | mean_score=0.113738 | path=/content/mvtec_raw/screw/t

In [8]:
!python -m src.training.train --category pill --loss composite --epochs 50 --lr 1e-4 --run_name ablation1_composite_pill --use_wandb

Using device: cuda
[pill] train: 267 | val: 25 | test: 142
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 205MB/s]
Encoder trainable params: 0 (freeze_encoder=True)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260808_093849-p2wf85jd
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ablation1_composite_pill
wandb: ⭐️ View project at https://wandb.ai

In [9]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/ablation1_composite_pill_best.pt

Loaded checkpoint from epoch 12 (val_auc at save time: 0.6786)
Config: loss=composite, freeze_encoder=True, encoder_backbone=resnet18
Test set size: 142

--- Aggregation: max ---
Overall test AUC-ROC: 0.4523
Per-defect-type AUC-ROC:
  faulty_imprint: 0.3239
  scratch: 0.3773
  contamination: 0.4116
  crack: 0.4442
  combined: 0.5195
  color: 0.5346
  pill_type: 0.6515

--- Aggregation: mean ---
Overall test AUC-ROC: 0.4856
Per-defect-type AUC-ROC:
  combined: 0.1916
  scratch: 0.4227
  contamination: 0.4924
  faulty_imprint: 0.5114
  color: 0.5368
  crack: 0.5393
  pill_type: 0.7727

Results saved to /content/drive/MyDrive/defectlens/results/ablation1_composite_pill_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/pill/test/color/013.png | defect_type: color | score: 0.5496
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/pill/test/combined/008.png | defect_type: combined | score: 0.3237
Figure(1500x500)

Vis

In [10]:
!python -m src.training.train --category screw --loss composite --no_freeze_encoder --epochs 50 --lr 1e-4 --run_name ablation2_finetune_screw --use_wandb

Using device: cuda
[screw] train: 320 | val: 24 | test: 136
Encoder trainable params: 2099712 (freeze_encoder=False)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run vgfha4fj (0.0s)
wandb: ⣻ setting up run vgfha4fj (0.0s)
wandb: ⣽ setting up run vgfha4fj (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260808_101102-vgfha4fj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ablation2_finetune_screw
wandb: ⭐️ View project at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens
wandb: 🚀 View run at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens/runs/vgfha4fj
Epoch 1/50 | train_loss: 0.7109 | val_mse: 0.0651 | val_au

In [11]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/ablation2_finetune_screw_best.pt

Loaded checkpoint from epoch 6 (val_auc at save time: 0.8889)
Config: loss=composite, freeze_encoder=False, encoder_backbone=resnet18
Test set size: 136

--- Aggregation: max ---
Overall test AUC-ROC: 0.8727
Per-defect-type AUC-ROC:
  manipulated_front: 0.6557
  scratch_neck: 0.8748
  scratch_head: 0.9343
  thread_top: 0.9371
  thread_side: 0.9614

--- Aggregation: mean ---
Overall test AUC-ROC: 1.0000
Per-defect-type AUC-ROC:
  manipulated_front: 1.0000
  scratch_head: 1.0000
  scratch_neck: 1.0000
  thread_side: 1.0000
  thread_top: 1.0000

Results saved to /content/drive/MyDrive/defectlens/results/ablation2_finetune_screw_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/screw/test/thread_side/002.png | defect_type: thread_side | score: 0.5744
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/screw/test/manipulated_front/007.png | defect_type: manipulated_front | score: 0.2844
Figure(1500x500)

Visualizing: 

In [12]:
!python -m src.training.train --category pill --loss mse --no_freeze_encoder --epochs 50 --lr 1e-4 --run_name ablation2_finetune_pill --use_wandb

Using device: cuda
[pill] train: 267 | val: 25 | test: 142
Encoder trainable params: 2099712 (freeze_encoder=False)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run t147lpok (0.0s)
wandb: ⣻ setting up run t147lpok (0.0s)
wandb: ⣽ setting up run t147lpok (0.0s)
wandb: ⣾ setting up run t147lpok (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260808_102842-t147lpok
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ablation2_finetune_pill
wandb: ⭐️ View project at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens
wandb: 🚀 View run at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens/runs/t147lpok
Epoch 1/50 | train_l

In [13]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/ablation2_finetune_pill_best.pt

Loaded checkpoint from epoch 28 (val_auc at save time: 0.6310)
Config: loss=mse, freeze_encoder=False, encoder_backbone=resnet18
Test set size: 142

--- Aggregation: max ---
Overall test AUC-ROC: 0.6269
Per-defect-type AUC-ROC:
  faulty_imprint: 0.4432
  scratch: 0.4545
  contamination: 0.4747
  color: 0.6602
  crack: 0.7831
  combined: 0.8214
  pill_type: 0.8788

--- Aggregation: mean ---
Overall test AUC-ROC: 0.6254
Per-defect-type AUC-ROC:
  combined: 0.5487
  scratch: 0.5750
  contamination: 0.5758
  faulty_imprint: 0.5994
  color: 0.6255
  crack: 0.6260
  pill_type: 1.0000

Results saved to /content/drive/MyDrive/defectlens/results/ablation2_finetune_pill_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/pill/test/color/006.png | defect_type: color | score: 0.6031
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/pill/test/contamination/008.png | defect_type: contamination | score: 0.2261
Figure(1500x500)


In [14]:
!python -m src.training.train --category screw --loss mse --encoder_backbone efficientnet_b0 --epochs 50 --lr 1e-3 --run_name ablation3_efficientnet_screw --use_wandb

Using device: cuda
[screw] train: 320 | val: 24 | test: 136
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 216MB/s]
Encoder trainable params: 0 (freeze_encoder=True)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run si6w70sj (0.0s)
wandb: ⣻ setting up run si6w70sj (0.0s)
wandb: ⣽ setting up run si6w70sj (0.0s)
wandb: ⣾ setting up run si6w70sj (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260808_105835-si6w70sj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ablation3_efficientn

In [15]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/ablation3_efficientnet_screw_best.pt

Loaded checkpoint from epoch 9 (val_auc at save time: 0.8056)
Config: loss=mse, freeze_encoder=True, encoder_backbone=efficientnet_b0
Test set size: 136

--- Aggregation: max ---
Overall test AUC-ROC: 0.8286
Per-defect-type AUC-ROC:
  manipulated_front: 0.7100
  thread_side: 0.8343
  scratch_head: 0.8371
  scratch_neck: 0.8694
  thread_top: 0.8900

--- Aggregation: mean ---
Overall test AUC-ROC: 0.0054
Per-defect-type AUC-ROC:
  manipulated_front: 0.0014
  thread_top: 0.0029
  thread_side: 0.0057
  scratch_neck: 0.0082
  scratch_head: 0.0086

Results saved to /content/drive/MyDrive/defectlens/results/ablation3_efficientnet_screw_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/screw/test/thread_side/002.png | defect_type: thread_side | score: 0.5097
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/screw/test/manipulated_front/007.png | defect_type: manipulated_front | score: 0.2661
Figure(1500x500)

Visualizi

In [16]:
!python -m src.training.train --category pill --loss mse --encoder_backbone efficientnet_b0 --epochs 50 --lr 1e-3 --run_name ablation3_efficientnet_pill --use_wandb

Using device: cuda
[pill] train: 267 | val: 25 | test: 142
Encoder trainable params: 0 (freeze_encoder=True)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run dg2senmj (0.0s)
wandb: ⣻ setting up run dg2senmj (0.0s)
wandb: ⣽ setting up run dg2senmj (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260808_111314-dg2senmj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ablation3_efficientnet_pill
wandb: ⭐️ View project at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens
wandb: 🚀 View run at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens/runs/dg2senmj
Epoch 1/50 | train_loss: 0.0541 | val_mse: 0.0278 | val_auc: 0.

In [17]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/ablation3_efficientnet_pill_best.pt

Loaded checkpoint from epoch 21 (val_auc at save time: 0.7976)
Config: loss=mse, freeze_encoder=True, encoder_backbone=efficientnet_b0
Test set size: 142

--- Aggregation: max ---
Overall test AUC-ROC: 0.6970
Per-defect-type AUC-ROC:
  scratch: 0.4591
  contamination: 0.5278
  faulty_imprint: 0.6080
  color: 0.7879
  crack: 0.8120
  pill_type: 0.8636
  combined: 0.9318

--- Aggregation: mean ---
Overall test AUC-ROC: 0.8034
Per-defect-type AUC-ROC:
  color: 0.6688
  crack: 0.7665
  scratch: 0.7727
  faulty_imprint: 0.7841
  contamination: 0.8232
  combined: 0.9773
  pill_type: 1.0000

Results saved to /content/drive/MyDrive/defectlens/results/ablation3_efficientnet_pill_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/pill/test/pill_type/001.png | defect_type: pill_type | score: 0.8534
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/pill/test/faulty_imprint/015.png | defect_type: faulty_imprint | score: 0.17

In [18]:
!python -m src.training.train --category screw --loss composite --no_freeze_encoder --encoder_backbone efficientnet_b0 --epochs 50 --lr 1e-4 --run_name combo_all_screw --use_wandb

Using device: cuda
[screw] train: 320 | val: 24 | test: 136
Encoder trainable params: 851808 (freeze_encoder=False)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run 0y2zyk0z (0.0s)
wandb: ⣻ setting up run 0y2zyk0z (0.0s)
wandb: ⣽ setting up run 0y2zyk0z (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260808_112813-0y2zyk0z
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run combo_all_screw
wandb: ⭐️ View project at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens
wandb: 🚀 View run at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens/runs/0y2zyk0z
Epoch 1/50 | train_loss: 0.7911 | val_mse: 0.0671 | val_auc: 0.9167 

In [19]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/combo_all_screw_best.pt

Loaded checkpoint from epoch 2 (val_auc at save time: 0.9537)
Config: loss=composite, freeze_encoder=False, encoder_backbone=efficientnet_b0
Test set size: 136

--- Aggregation: max ---
Overall test AUC-ROC: 0.8000
Per-defect-type AUC-ROC:
  manipulated_front: 0.5814
  thread_side: 0.7600
  thread_top: 0.8629
  scratch_head: 0.8914
  scratch_neck: 0.8993

--- Aggregation: mean ---
Overall test AUC-ROC: 1.0000
Per-defect-type AUC-ROC:
  manipulated_front: 1.0000
  scratch_head: 1.0000
  scratch_neck: 1.0000
  thread_side: 1.0000
  thread_top: 1.0000

Results saved to /content/drive/MyDrive/defectlens/results/combo_all_screw_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/screw/test/scratch_neck/016.png | defect_type: scratch_neck | score: 0.7449
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/screw/test/manipulated_front/007.png | defect_type: manipulated_front | score: 0.4301
Figure(1500x500)

Visualizing: 

In [20]:
!python -m src.training.train --category pill --loss composite --no_freeze_encoder --encoder_backbone efficientnet_b0 --epochs 50 --lr 1e-4 --run_name combo_all_pill --use_wandb

Using device: cuda
[pill] train: 267 | val: 25 | test: 142
Encoder trainable params: 851808 (freeze_encoder=False)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: chsriramsatvik20 (chsriramsatvik20-iit-guwahati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run 09qcm066 (0.0s)
wandb: ⣻ setting up run 09qcm066 (0.0s)
wandb: ⣽ setting up run 09qcm066 (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/defectlens/wandb/run-20260808_114152-09qcm066
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run combo_all_pill
wandb: ⭐️ View project at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens
wandb: 🚀 View run at https://wandb.ai/chsriramsatvik20-iit-guwahati/defectlens/runs/09qcm066
Epoch 1/50 | train_loss: 1.1047 | val_mse: 0.1207 | val_auc: 0.2857 | 

In [21]:
!python -m src.eval.evaluate --checkpoint /content/drive/MyDrive/defectlens/checkpoints/combo_all_pill_best.pt

Loaded checkpoint from epoch 47 (val_auc at save time: 0.5595)
Config: loss=composite, freeze_encoder=False, encoder_backbone=efficientnet_b0
Test set size: 142

--- Aggregation: max ---
Overall test AUC-ROC: 0.4924
Per-defect-type AUC-ROC:
  scratch: 0.3659
  contamination: 0.3712
  faulty_imprint: 0.4148
  color: 0.4848
  combined: 0.5065
  crack: 0.6281
  pill_type: 0.8182

--- Aggregation: mean ---
Overall test AUC-ROC: 0.7697
Per-defect-type AUC-ROC:
  color: 0.6494
  scratch: 0.7023
  faulty_imprint: 0.7102
  crack: 0.7417
  contamination: 0.7955
  combined: 0.9773
  pill_type: 1.0000

Results saved to /content/drive/MyDrive/defectlens/results/combo_all_pill_test_results.json

Visualizing: best_detected_defect | path: /content/mvtec_raw/pill/test/pill_type/000.png | defect_type: pill_type | score: 0.7827
Figure(1500x500)

Visualizing: worst_detected_defect (likely missed) | path: /content/mvtec_raw/pill/test/combined/015.png | defect_type: combined | score: 0.3120
Figure(1500x500

In [22]:
!ls /content/drive/MyDrive/defectlens/checkpoints/
!ls /content/drive/MyDrive/defectlens/results/

ablation1_composite_pill_best.pt
ablation1_composite_pill_history.json
ablation1_composite_screw_best.pt
ablation1_composite_screw_history.json
ablation1_composite_screw_lowlr_best.pt
ablation1_composite_screw_lowlr_history.json
ablation2_finetune_pill_best.pt
ablation2_finetune_pill_history.json
ablation2_finetune_screw_best.pt
ablation2_finetune_screw_history.json
ablation3_efficientnet_pill_best.pt
ablation3_efficientnet_pill_history.json
ablation3_efficientnet_screw_best.pt
ablation3_efficientnet_screw_history.json
baseline_mse_pill_best.pt
baseline_mse_pill_history.json
baseline_mse_screw_best.pt
baseline_mse_screw_history.json
combo_all_pill_best.pt
combo_all_pill_history.json
combo_all_screw_best.pt
combo_all_screw_history.json
smoke_test_best.pt
smoke_test_history.json
test.txt
ablation1_composite_pill_test_results.json
ablation1_composite_screw_lowlr_test_results.json
ablation2_finetune_pill_test_results.json
ablation2_finetune_screw_test_results.json
ablation3_efficientnet_pi